In [1]:
!pip -q install awscli nilearn nibabel pandas

import os
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [2]:
!mkdir -p /content/ds006072

In [3]:
!aws s3 ls --no-sign-request s3://openneuro.org/ds006072/ > bucket.txt

In [4]:
with open("bucket.txt", "r") as f:
    lines = f.readlines()

subjects = []

for line in lines:
    if "PRE sub-" in line:
        subjects.append(line.strip().split()[-1].rstrip("/"))

subjects = sorted(subjects)

print("Subjects found:", len(subjects))
print(subjects)

Subjects found: 11
['sub-P1', 'sub-P1R', 'sub-P2', 'sub-P3', 'sub-P3R', 'sub-P4', 'sub-P4R', 'sub-P5', 'sub-P5R', 'sub-P6', 'sub-P7']


In [5]:
subject = subjects[0]
print("Inspecting:", subject)

!aws s3 ls --recursive --no-sign-request s3://openneuro.org/ds006072/{subject}/ | head -50

Inspecting: sub-P1
2025-03-31 16:36:27       2199 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t1mpr0p9mm_rec-NDNORM_run-1_T1w.json
2025-03-31 16:36:28    8963163 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t1mpr0p9mm_rec-NDNORM_run-1_T1w.nii.gz
2025-03-31 16:36:28       2463 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t1mpr0p9mm_rec-NDNORM_run-2_T1w.json
2025-03-31 16:36:28    8963274 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t1mpr0p9mm_rec-NDNORM_run-2_T1w.nii.gz
2025-03-31 16:36:29       2217 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t2spc0p9mmiso_rec-NDNORM_run-1_T2w.json
2025-03-31 16:36:29    8276862 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t2spc0p9mmiso_rec-NDNORM_run-1_T2w.nii.gz
2025-03-31 16:36:29       2511 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t2spc0p9mmiso_rec-NDNORM_run-2_T2w.json
2025-03-31 16:36:30    8276967 ds006072/sub-P1/ses-1/anat/sub-P1_ses-1_acq-t2spc0p9mmiso_rec-NDNORM_run-2_T2w.nii.gz
2025-03-31 16:36:30        473 ds006072/sub-P1/ses-1/dwi/sub-P1_s

In [6]:
# Pivot Note:
# Initial plan was to use all resting-state and task-fMRI files.
# Inspection showed the dataset contains very large multi-echo raw files,
# making full download impractical in Colab.
# Scope revised to a feasible pipeline using selected single-echo resting-state files.

In [7]:
selected_echo = "echo-2"
selected_task = "BOLDREST"

print("Pivot strategy:")
print("Task type:", selected_task)
print("Echo:", selected_echo)

Pivot strategy:
Task type: BOLDREST
Echo: echo-2


In [8]:
subject = subjects[0]

!aws s3 ls --recursive --no-sign-request s3://openneuro.org/ds006072/{subject}/func/ | grep {selected_task} | grep {selected_echo} | head -20

In [9]:
!mkdir -p /content/ds006072/sample_func

In [10]:
test_file = "sub-P1/ses-1/func/sub-P1_ses-1_task-BOLDREST1_dir-AP_run-1_echo-2_part-mag_bold.nii"

!aws s3 cp --no-sign-request \
s3://openneuro.org/ds006072/{test_file} \
/content/ds006072/sample_func/

download: s3://openneuro.org/ds006072/sub-P1/ses-1/func/sub-P1_ses-1_task-BOLDREST1_dir-AP_run-1_echo-2_part-mag_bold.nii to ds006072/sample_func/sub-P1_ses-1_task-BOLDREST1_dir-AP_run-1_echo-2_part-mag_bold.nii


In [11]:
import nibabel as nib
import os

fp = "/content/ds006072/sample_func/sub-P1_ses-1_task-BOLDREST1_dir-AP_run-1_echo-2_part-mag_bold.nii"

img = nib.load(fp)

print("Shape:", img.shape)
print("File size (GB):", round(os.path.getsize(fp)/1e9, 2))

Shape: (110, 110, 72, 513)
File size (GB): 0.89


In [12]:
print("Notebook 1 complete.")
print("Pivot successful: single-echo resting-state pipeline established.")
print("Next step: batch download and connectivity extraction.")

Notebook 1 complete.
Pivot successful: single-echo resting-state pipeline established.
Next step: batch download and connectivity extraction.
